# Pipeline Inspection: Processing Pipeline (TomoSAR Raw Product Generation)

This notebook executes the `ProcessingPipeline` (`pipelines/processing_pipeline/pipeline.py`) stage by stage and verifies that each step produces the contracted output before the next stage consumes it.

The processing pipeline builds the **raw tomographic SAR products** that feed every downstream training and inference run. From a stack of co-registered F-SAR single-look-complex (SLC) acquisitions it (1) dumps the resolved configuration, (2) focuses two SAR **tomograms** along the elevation / height axis via Capon beamforming run in parallel PyRat worker processes — a *full* (parameter) variant and a *reduced* variant — together with their digital elevation models (DEM), and (3) forms the **interferometric stack** (primary SLC, secondary SLCs, and flat-earth / DEM-deramped interferograms) for the reduced acquisition. A metadata record and a dataset-layout manifest are written alongside.

**What this notebook verifies, stage by stage:**

- The resolved `ProcessingConfiguration` is serialised to disk before any heavy computation runs.
- The `full` and `reduced` `TomogramVariant` tuples resolve to the correct stack identifiers, configs and tags, and the azimuth crop is subdivided into worker subsections that tile the crop with no gaps or overlaps.
- Each PyRat worker job carries a self-consistent geometry (crop, height range, beamforming and filter arguments) and writes one HDF5 partial holding a `tomogram` cube and a `DEM` raster.
- Concatenation reassembles the partials along azimuth into a single elevation-resolved tomogram cube and a single DEM with the expected combined shape.
- The interferogram builder returns a complex64 primary SLC, a stack of complex64 secondaries, and a stack of amplitude-weighted, DEM-deramped interferograms of identical spatial shape.
- Every artifact path, metadata file, and the `dataset.json` layout manifest exist and are mutually consistent with the config tags.

> This notebook does **not** train or predict. It is a correctness harness: run it once to establish a baseline of shapes, ranges and complex-valued statistics, change a processing component, run it again, and the assertions either still pass or pinpoint the regression.

**Note on execution:** the focusing stages require a working PyRat installation and the F-SAR acquisition stack on disk (`config.paths.pyrat_directory`, `config.paths.main_directory`). Where those are unavailable the heavy cells will raise on the PyRat import / data load; every cell is nonetheless written to run unmodified once the data is in place.

In [ ]:
from __future__ import annotations

import sys
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configuration.processing_config            import ProcessingConfiguration, TomogramConfiguration, ParallelConfiguration, PathConfiguration
from pipelines.processing_pipeline.pipeline      import ProcessingPipeline, TomogramVariant
from pipelines.processing_pipeline.artifacts     import ArtifactRegistry
from pipelines.processing_pipeline.interferogram import InterferogramBuilder
from pipelines.processing_pipeline.metadata      import MetadataManager
from pipelines.processing_pipeline.tomogram      import TomogramProcessor
from pipelines.processing_pipeline.tomogram_worker import PyRatJob, run_pyrat
from tools.crop_region                           import CropRegion
from tools.logger                                import Logger

mpl.rcParams.update({
    "font.family"        : "serif",
    "font.size"          : 11,
    "axes.labelsize"     : 12,
    "axes.titlesize"     : 13,
    "legend.fontsize"    : 10,
    "xtick.labelsize"    : 10,
    "ytick.labelsize"    : 10,
    "figure.dpi"         : 150,
    "savefig.dpi"        : 300,
    "savefig.bbox"       : "tight",
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
})

figure_directory = Path("figures/inspect_processing_pipeline")
figure_directory.mkdir(parents=True, exist_ok=True)

intensity_colormap = "inferno"
phase_colormap     = "twilight"
height_colormap    = "viridis"

processing_crop = CropRegion(
    azimuth_start = 0,
    azimuth_end   = 1500,
    range_start   = 0,
    range_end     = 600,
)

processing_configuration = ProcessingConfiguration(
    crop          = processing_crop,
    input_configs = TomogramConfiguration(),
    parallel      = ParallelConfiguration(tomogram_workers=None, pyrat_threads=15),
    paths         = PathConfiguration(),
)

processing_logger = Logger(
    log_dir = str(Path(processing_configuration.paths.run_directory) / "logs"),
    name    = "processing_inspection",
    level   = "INFO",
)

print("Project root             :", project_root)
print("Run directory            :", processing_configuration.paths.run_directory)
print("Data directory           :", processing_configuration.paths.data_directory)
print("Metadata directory       :", processing_configuration.paths.metadata_directory)
print("Dataset type             :", processing_configuration.dataset_type)
print("Full stack identifier    :", processing_configuration.full_stack_identifier)
print("Reduced stack identifier :", processing_configuration.reduced_stack_identifier)
print("Global crop (a0,a1,r0,r1):", processing_configuration.crop.as_tuple())
print("Crop azimuth size  [pix] :", processing_configuration.crop.azimuth_size)
print("Crop range size    [pix] :", processing_configuration.crop.range_size)
print("Height range       [m]   :", processing_configuration.input_configs.height_range)
print("Beamforming method       :", processing_configuration.input_configs.beamforming_method)
print("Filter method            :", processing_configuration.input_configs.filter_method)
print("Filter window     [px,px]:", processing_configuration.input_configs.filter_arguments.get("win"))
print("Max azimuth subsection   :", processing_configuration.input_configs.max_crop_azimuth_width)
print("Tomogram tag             :", processing_configuration.tomogram_tag)
print("Parameter tag            :", processing_configuration.parameter_tag)

## Pipeline Overview

`ProcessingPipeline.run()` orchestrates the following ordered sequence. Each arrow is a handoff of a concrete artifact (a NumPy `.npy`, an HDF5 partial, or a metadata file) from one component to the next.

```
ProcessingConfiguration
  └─► Stage 1  MetadataManager.save_pipeline_configuration()      →  config_state_<tag>.json
        └─► Stage 2  _resolve_variant() + TomogramProcessor._divide_crop()  →  TomogramVariant + azimuth subsections
              └─► Stage 3  TomogramProcessor._dispatch_workers() → run_pyrat() →  per-subsection HDF5 partials (tomogram + DEM)
                    └─► Stage 4  TomogramProcessor._concatenate()           →  combined tomogram cube + combined DEM
                          └─► Stage 5  _stage_tomogram("full")             →  tomogram_full.npy, dem_full.npy, meta_tomogram_full
                                └─► Stage 6  _stage_tomogram("reduced")     →  tomogram_reduced.npy, dem_reduced.npy, meta_tomogram_reduced
                                      └─► Stage 7  InterferogramBuilder.build()  →  primary SLC, secondaries, interferograms (complex64)
                                            └─► Stage 8  _stage_inputs() persist  →  primary/secondaries/interferograms .npy + meta_inputs
                                                  └─► Stage 9  MetadataManager.save_dataset_layout()  →  dataset.json
```

The pipeline focuses **two tomograms** (a *full* / parameter variant keyed on `full_stack_identifier`, and a *reduced* variant keyed on `reduced_stack_identifier`) before forming the interferometric stack for the reduced acquisition only. Stages 2–4 expose the internal mechanics of a single `_stage_tomogram` call; Stages 5 and 6 wrap them into the two end-to-end variant calls that `run()` actually issues, in order.

| Stage | Callable | Input | Output |
|-------|----------|-------|--------|
| 1 | `MetadataManager.save_pipeline_configuration()` | `ProcessingConfiguration` | `config_state_<tomogram_tag>.json` (`Path`) |
| 2 | `ProcessingPipeline._resolve_variant()` + `TomogramProcessor._divide_crop()` | variant literal, `TomogramConfiguration` | `TomogramVariant`, list of azimuth subsection crops |
| 3 | `TomogramProcessor._dispatch_workers()` → `run_pyrat(PyRatJob)` | subsection crops, stack id, tomogram config | per-subsection HDF5 partials under `TOMO/TOMO-SR` |
| 4 | `TomogramProcessor._concatenate()` | temporary directory of HDF5 partials | `(combined_dem, combined_tomogram)` arrays |
| 5 | `ProcessingPipeline._stage_tomogram("full")` | variant literal `"full"` | `(tomogram_full_path, dem_full_path)` + metadata |
| 6 | `ProcessingPipeline._stage_tomogram("reduced")` | variant literal `"reduced"` | `(tomogram_reduced_path, dem_reduced_path)` + metadata |
| 7 | `InterferogramBuilder.build()` | crop tuple | `(primary, secondaries, interferograms)` complex64 |
| 8 | `ProcessingPipeline._stage_inputs()` | — | `(primary_path, secondaries_path, interferograms_path)` + metadata |
| 9 | `MetadataManager.save_dataset_layout()` | — | `dataset.json` (`Path`) |

**Component instances** used throughout (constructed once, matching the pipeline's `__init__`):

In [ ]:
artifact_registry      = ArtifactRegistry  (processing_configuration, logger=processing_logger)
metadata_manager       = MetadataManager   (processing_configuration, logger=processing_logger)
tomogram_processor     = TomogramProcessor (processing_configuration, logger=processing_logger)
interferogram_builder  = InterferogramBuilder(processing_configuration, logger=processing_logger)

print("artifact_registry     :", type(artifact_registry).__name__)
print("metadata_manager      :", type(metadata_manager).__name__)
print("tomogram_processor    :", type(tomogram_processor).__name__)
print("interferogram_builder :", type(interferogram_builder).__name__)
print()
print("Artifact filenames resolved from config tags:")
for artifact_name, artifact_filename in artifact_registry.artifact_filenames().items():
    print(f"  {artifact_name:<24}: {artifact_filename}")

---
## Stage 1: Persist Resolved Configuration

**Callable:** `MetadataManager.save_pipeline_configuration()`
**Input:** the `ProcessingConfiguration` instance.
**Output:** a `Path` to `config_state_<tomogram_tag>.json` in the metadata directory; the directory tree (`data/`, `meta/`, `tmp/`) is created as a side effect.

This stage has no signal-model transformation. It freezes the *exact* parameters that govern every later stage so the run is reproducible. The serialised tag is

$$\text{tomogram\_tag} = \langle\text{crop identifier}\rangle\;\Vert\;\langle\text{reduced stack id}\rangle\;\Vert\;\langle\text{tomogram output tag}\rangle$$

where the crop identifier joins $(a_0, a_1, r_0, r_1)$ with the literal `a`.

> **What you should see:** a JSON file whose name starts with `config_state_` and ends with `.json`, sitting under `processing_configuration.paths.metadata_directory`. The reloaded JSON must contain a `crop` block equal to `processing_configuration.crop.as_tuple()`, a `dataset_type` of `"FSAR"`, and the same `full_stack_identifier` / `reduced_stack_identifier` as the live config. The three working directories (`data/`, `meta/`, `tmp/`) must now exist on disk.

In [ ]:
configuration_dump_path = metadata_manager.save_pipeline_configuration()
print("Configuration dump path :", configuration_dump_path)

In [ ]:
reloaded_configuration = json.loads(Path(configuration_dump_path).read_text(encoding="utf-8"))

print("Dump file name           :", configuration_dump_path.name)
print("Dump exists on disk      :", configuration_dump_path.exists())
print("Top-level JSON keys      :", sorted(reloaded_configuration.keys()))
print()
print("Serialised crop          :", reloaded_configuration["crop"])
print("Serialised dataset_type  :", reloaded_configuration["dataset_type"])
print("Serialised full stack id :", reloaded_configuration["full_stack_identifier"])
print("Serialised reduced id    :", reloaded_configuration["reduced_stack_identifier"])
print()
print("Directory existence:")
for directory_label, directory_path in (
    ("data_directory",     processing_configuration.paths.data_directory),
    ("metadata_directory", processing_configuration.paths.metadata_directory),
    ("temporary_directory",processing_configuration.paths.temporary_directory),
):
    print(f"  {directory_label:<20}: exists={directory_path.exists()}  path={directory_path}")

In [ ]:
serialised_crop      = reloaded_configuration["crop"]
serialised_crop_tuple = (
    serialised_crop["azimuth_start"],
    serialised_crop["azimuth_end"],
    serialised_crop["range_start"],
    serialised_crop["range_end"],
) if isinstance(serialised_crop, dict) else tuple(serialised_crop)

stage_1_checks = [
    ("Dump path is a file on disk",            configuration_dump_path.is_file()),
    ("Dump file name has config_state prefix", configuration_dump_path.name.startswith("config_state_")),
    ("Dump file name has .json suffix",        configuration_dump_path.suffix == ".json"),
    ("Dump sits in metadata directory",        configuration_dump_path.parent == processing_configuration.paths.metadata_directory),
    ("Serialised crop equals live crop",       serialised_crop_tuple == processing_configuration.crop.as_tuple()),
    ("Serialised dataset_type is FSAR",        reloaded_configuration["dataset_type"] == processing_configuration.dataset_type),
    ("Serialised full stack id matches",       reloaded_configuration["full_stack_identifier"] == processing_configuration.full_stack_identifier),
    ("Serialised reduced id matches",          reloaded_configuration["reduced_stack_identifier"] == processing_configuration.reduced_stack_identifier),
    ("data directory created",                 processing_configuration.paths.data_directory.exists()),
    ("metadata directory created",             processing_configuration.paths.metadata_directory.exists()),
    ("temporary directory created",            processing_configuration.paths.temporary_directory.exists()),
]
for stage_1_description, stage_1_condition in stage_1_checks:
    print(f"[{'PASS' if stage_1_condition else 'FAIL'}] {stage_1_description}")

### Common Mistakes — Stage 1: Persist Resolved Configuration

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Two runs overwrite the same dump | `run_subdirectory` left at a fixed string instead of the timestamped default | Print `processing_configuration.paths.run_directory`; the default `__post_init__` appends `run_<tag>_<timestamp>` |
| `crop` block serialised as `null` or a string | `CropRegion` is not a dataclass instance, so `asdict` cannot recurse | Confirm `isinstance(processing_configuration.crop, CropRegion)` and that it survives `dataclasses.asdict` |
| Tags in the file name disagree with downstream artifact names | `tomogram_output_tag` / `parameter_output_tag` edited after the dump but before focusing | Compare `configuration_dump_path.name` against `artifact_registry.artifact_filenames()` |
| `PermissionError` on write | `main_directory` points at a read-only mount | Verify the metadata directory exists and is writable before this stage |
| Non-reproducible later stages despite a saved config | a parameter (e.g. `pyrat_threads`) read from the environment instead of the serialised config | Diff the reloaded JSON against the live config field by field |

**Passing to Stage 2:** `configuration_dump_path` — `Path`, the frozen configuration snapshot; the live `processing_configuration` and the constructed `tomogram_processor` now drive variant resolution and crop subdivision.

---
## Stage 2: Resolve Variant and Subdivide the Azimuth Crop

**Callable:** `ProcessingPipeline._resolve_variant(variant)` + `TomogramProcessor._divide_crop(tomogram_config)`
**Input:** the variant literal (`"full"` or `"reduced"`) and the corresponding `TomogramConfiguration`.
**Output:** a `TomogramVariant(stack_identifier, tomogram_config, identifier_tag)` and a list of azimuth-tiled subsection crops $\{(a_i, a_{i+1}, r_0, r_1)\}$.

The full azimuth extent $[a_0, a_1)$ is partitioned into contiguous tiles of width at most $W_{\max}$ (`max_crop_azimuth_width`):

$$a_0 = c_0 < c_1 < \dots < c_M = a_1, \qquad c_{i+1} - c_i \le W_{\max}, \qquad M = \left\lceil \frac{a_1 - a_0}{W_{\max}} \right\rceil$$

Each tile keeps the full range extent $[r_0, r_1)$. The tiling is exact: the union of the tiles is the original crop, and no two tiles overlap. This subdivision bounds peak memory per worker, since Capon beamforming forms a covariance matrix per range/azimuth window.

> **What you should see:** for the `"full"` variant, `TomogramVariant.stack_identifier == full_stack_identifier` and `identifier_tag == parameter_tag`; for `"reduced"`, the reduced identifier and `tomogram_tag`. With an azimuth size of 1500 and `max_crop_azimuth_width = 1000`, the crop subdivides into **2** subsections of widths 1000 and 500. The first subsection starts at `azimuth_start`, the last ends at `azimuth_end`, every interior boundary is shared, and every tile carries the unchanged range bounds.

In [ ]:
bound_pipeline_for_resolution = ProcessingPipeline.__new__(ProcessingPipeline)
bound_pipeline_for_resolution.config = processing_configuration

resolved_full_variant    = bound_pipeline_for_resolution._resolve_variant("full")
resolved_reduced_variant = bound_pipeline_for_resolution._resolve_variant("reduced")

full_subsection_crops    = tomogram_processor._divide_crop(resolved_full_variant.tomogram_config)
reduced_subsection_crops = tomogram_processor._divide_crop(resolved_reduced_variant.tomogram_config)

In [ ]:
print("Resolved FULL variant:")
print("  stack_identifier :", resolved_full_variant.stack_identifier)
print("  identifier_tag   :", resolved_full_variant.identifier_tag)
print("  beamforming      :", resolved_full_variant.tomogram_config.beamforming_method)
print("  height_range [m] :", resolved_full_variant.tomogram_config.height_range)
print()
print("Resolved REDUCED variant:")
print("  stack_identifier :", resolved_reduced_variant.stack_identifier)
print("  identifier_tag   :", resolved_reduced_variant.identifier_tag)
print()
print("FULL subsection crops    :", full_subsection_crops)
print("REDUCED subsection crops :", reduced_subsection_crops)
print()
print("Number of FULL subsections    :", len(full_subsection_crops))
print("Number of REDUCED subsections :", len(reduced_subsection_crops))
print()
print("Per-subsection azimuth widths :", [c[1] - c[0] for c in full_subsection_crops])
print("Max azimuth width limit       :", resolved_full_variant.tomogram_config.max_crop_azimuth_width)

In [ ]:
subsection_starts = np.array([crop[0] for crop in full_subsection_crops])
subsection_ends   = np.array([crop[1] for crop in full_subsection_crops])
subsection_widths = subsection_ends - subsection_starts

figure, axis = plt.subplots(figsize=(8.0, 2.6))
for subsection_index, (start, end) in enumerate(zip(subsection_starts, subsection_ends)):
    axis.barh(0, end - start, left=start, height=0.5, edgecolor="black", linewidth=0.8, color=plt.cm.viridis(subsection_index / max(1, len(full_subsection_crops) - 1)))
    axis.text((start + end) / 2.0, 0.0, f"#{subsection_index}\n[{start}, {end})", ha="center", va="center", fontsize=9, color="white")

axis.set_xlim(processing_configuration.crop.azimuth_start - 20, processing_configuration.crop.azimuth_end + 20)
axis.set_ylim(-0.6, 0.6)
axis.set_yticks([])
axis.set_xlabel("Azimuth sample index [pixel]")
axis.set_title("Stage 2 — Crop Subdivision: azimuth tiling into PyRat worker subsections")
figure.savefig(figure_directory / "stage_2_resolve_and_subdivide.pdf")
plt.show()

In [ ]:
reconstructed_union_start = int(subsection_starts.min())
reconstructed_union_end   = int(subsection_ends.max())
interior_boundaries_match = bool(np.all(subsection_ends[:-1] == subsection_starts[1:])) if len(full_subsection_crops) > 1 else True
range_bounds_preserved    = all(crop[2] == processing_configuration.crop.range_start and crop[3] == processing_configuration.crop.range_end for crop in full_subsection_crops)
expected_subsection_count = -(-processing_configuration.crop.azimuth_size // processing_configuration.input_configs.max_crop_azimuth_width)

stage_2_checks = [
    ("FULL stack id equals full_stack_identifier",       resolved_full_variant.stack_identifier == processing_configuration.full_stack_identifier),
    ("FULL tag equals parameter_tag",                    resolved_full_variant.identifier_tag == processing_configuration.parameter_tag),
    ("REDUCED stack id equals reduced_stack_identifier", resolved_reduced_variant.stack_identifier == processing_configuration.reduced_stack_identifier),
    ("REDUCED tag equals tomogram_tag",                  resolved_reduced_variant.identifier_tag == processing_configuration.tomogram_tag),
    ("Subsection count matches ceil(width/limit)",       len(full_subsection_crops) == expected_subsection_count),
    ("First subsection starts at azimuth_start",         reconstructed_union_start == processing_configuration.crop.azimuth_start),
    ("Last subsection ends at azimuth_end",              reconstructed_union_end == processing_configuration.crop.azimuth_end),
    ("Interior boundaries are shared (no gap/overlap)",  interior_boundaries_match),
    ("All tile widths within max_crop_azimuth_width",    bool(np.all(subsection_widths <= processing_configuration.input_configs.max_crop_azimuth_width))),
    ("Range bounds preserved across all tiles",          range_bounds_preserved),
]
for stage_2_description, stage_2_condition in stage_2_checks:
    print(f"[{'PASS' if stage_2_condition else 'FAIL'}] {stage_2_description}")

### Common Mistakes — Stage 2: Resolve Variant and Subdivide

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Full and reduced tomograms come out identical | `_resolve_variant` returns the same stack id for both — `full_stack_identifier == reduced_stack_identifier` | Print both `TomogramVariant.stack_identifier`; they must differ |
| Wrong artifact tag on the focused product | `output_config` vs `input_configs` swapped, so the full variant uses the reduced config | Compare `identifier_tag` against `parameter_tag` (full) and `tomogram_tag` (reduced) |
| A thin seam of missing azimuth between tiles | off-by-one in the `while` loop (`<` vs `<=`), leaving the last column unfocused | Check `subsection_ends[-1] == azimuth_end` and shared interior boundaries |
| Tiles overlap and double-count azimuth | `current_azimuth` advanced by `max_width` instead of `next_azimuth` | Assert `subsection_ends[:-1] == subsection_starts[1:]` |
| Worker runs out of memory | `max_crop_azimuth_width` too large for the Capon covariance window | Reduce `max_crop_azimuth_width`; widths print above |
| Range crop silently dropped | a tile rebuilt without `range_start` / `range_end` | Confirm `range_bounds_preserved` is True |

**Passing to Stage 3:** `resolved_reduced_variant` — `TomogramVariant`; `reduced_subsection_crops` — `list[tuple[int,int,int,int]]`, the azimuth-tiled crops one PyRat worker will focus each.

---
## Stage 3: Dispatch Parallel PyRat Focusing Workers

**Callable:** `TomogramProcessor._dispatch_workers(subsections, stack_identifier, tomogram_config, temporary_directory)` → `run_pyrat(PyRatJob)`
**Input:** the list of azimuth subsection crops, the stack identifier, the variant `TomogramConfiguration`, and a fresh temporary directory.
**Output:** one HDF5 partial per subsection under `<temp>/TOMO/TOMO-SR`, each holding a focused `tomogram` cube and a `DEM` raster. Returns `None`; the partials are the side effect.

Each worker focuses the SAR signal along the **elevation / height** axis. For a range–azimuth pixel the co-registered stack of $N$ acquisitions samples the vertical reflectivity through their interferometric baselines. Capon (minimum-variance distortionless response) beamforming estimates the reflectivity power at height $h$ as

$$P_{\text{Capon}}(h) = \frac{1}{\mathbf{a}(h)^{\mathsf H}\,\hat{\mathbf R}^{-1}\,\mathbf{a}(h)}, \qquad \hat{\mathbf R} = \frac{1}{|\Omega|}\sum_{\Omega} \mathbf{s}\,\mathbf{s}^{\mathsf H}$$

where $\mathbf{a}(h)$ is the elevation steering vector built from the acquisition baselines, $\hat{\mathbf R}$ is the sample covariance estimated over the boxcar window $\Omega$ (`filter_arguments["win"]`), and $h$ ranges over `height_range`. The result is an elevation-resolved intensity cube; the DEM is the focused terrain reference.

> **What you should see:** $M$ `PyRatJob` instances (one per subsection from Stage 2), each carrying the **same** beamforming method, filter window, height range and stack identifier, but a **distinct** zero-padded `suffix` (`0000`, `0001`, …) and its own subsection crop. The resolved worker count is `min(M, cores // pyrat_threads)` and never exceeds $M$. After dispatch the `TOMO/TOMO-SR` directory holds exactly $M$ HDF5 files, each exposing a `tomogram` and a `DEM` dataset; the tomogram's azimuth extent equals its subsection width.

In [ ]:
inspection_temporary_directory = tomogram_processor._create_temp()

reduced_pyrat_jobs = []
parent_system_path = list(sys.path)
for subsection_index, subsection_crop in enumerate(reduced_subsection_crops):
    reduced_pyrat_jobs.append(PyRatJob(
        pyrat_root_path       = str(processing_configuration.paths.pyrat_directory),
        crop_tuple            = subsection_crop,
        suffix                = f"{subsection_index:04d}",
        fusar_project_path    = resolved_reduced_variant.tomogram_config.fusar_project_path,
        stack_identifier      = resolved_reduced_variant.stack_identifier,
        base_directory        = resolved_reduced_variant.tomogram_config.base_directory,
        polarisation          = resolved_reduced_variant.tomogram_config.polarisation,
        track_selection       = resolved_reduced_variant.tomogram_config.track_selection,
        height_range          = resolved_reduced_variant.tomogram_config.height_range,
        filter_method         = resolved_reduced_variant.tomogram_config.filter_method,
        filter_arguments      = resolved_reduced_variant.tomogram_config.filter_arguments,
        beamforming_method    = resolved_reduced_variant.tomogram_config.beamforming_method,
        beamforming_arguments = resolved_reduced_variant.tomogram_config.beamforming_arguments,
        output_directory      = str(inspection_temporary_directory),
        apply_resampling      = resolved_reduced_variant.tomogram_config.apply_resampling,
        apply_presumming      = resolved_reduced_variant.tomogram_config.apply_presumming,
        pyrat_threads         = processing_configuration.parallel.pyrat_threads,
        parent_sys_path       = parent_system_path,
    ))

resolved_worker_count = processing_configuration.parallel.resolve_workers(len(reduced_pyrat_jobs))

tomogram_processor._dispatch_workers(
    subsections         = reduced_subsection_crops,
    stack_identifier    = resolved_reduced_variant.stack_identifier,
    tomogram_config     = resolved_reduced_variant.tomogram_config,
    temporary_directory = inspection_temporary_directory,
)

In [ ]:
partial_files_directory = inspection_temporary_directory / "TOMO" / "TOMO-SR"
partial_file_paths      = sorted(partial_files_directory.iterdir()) if partial_files_directory.exists() else []

print("Temporary directory      :", inspection_temporary_directory)
print("Available cores          :", processing_configuration.parallel.available_cores())
print("PyRat threads per worker :", processing_configuration.parallel.pyrat_threads)
print("Resolved worker count    :", resolved_worker_count)
print("Number of dispatched jobs:", len(reduced_pyrat_jobs))
print()
print("Per-job geometry (suffix -> crop):")
for pyrat_job in reduced_pyrat_jobs:
    print(f"  suffix={pyrat_job.suffix}  crop={pyrat_job.crop_tuple}  method={pyrat_job.beamforming_method}  range={pyrat_job.height_range}  win={pyrat_job.filter_arguments.get('win')}")
print()
print("HDF5 partials written    :", len(partial_file_paths))
for partial_file_path in partial_file_paths:
    with h5py.File(str(partial_file_path), "r") as hdf5_file:
        print(f"  {partial_file_path.name}: tomogram={hdf5_file['tomogram'].shape} {hdf5_file['tomogram'].dtype}  DEM={hdf5_file['DEM'].shape} {hdf5_file['DEM'].dtype}")

In [ ]:
if partial_file_paths:
    with h5py.File(str(partial_file_paths[0]), "r") as hdf5_file:
        first_partial_tomogram = hdf5_file["tomogram"][:]
        first_partial_dem      = hdf5_file["DEM"][:]

    first_partial_intensity = np.abs(first_partial_tomogram).astype(np.float64)
    elevation_axis_length   = first_partial_intensity.shape[0]
    mean_elevation_profile  = first_partial_intensity.reshape(elevation_axis_length, -1).mean(axis=1)
    height_axis             = np.linspace(resolved_reduced_variant.tomogram_config.height_range[0], resolved_reduced_variant.tomogram_config.height_range[1], elevation_axis_length)

    figure, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    axes[0].plot(mean_elevation_profile, height_axis, color="#1b3b6f", linewidth=1.4)
    axes[0].set_xlabel("Mean focused intensity [linear]")
    axes[0].set_ylabel("Elevation / height [m]")
    axes[0].set_title("Stage 3 — Subsection #0: scene-averaged elevation profile")

    range_azimuth_intensity = first_partial_intensity.max(axis=0)
    intensity_image = axes[1].imshow(range_azimuth_intensity, aspect="auto", cmap=intensity_colormap, origin="lower")
    axes[1].set_xlabel("Azimuth sample [pixel]")
    axes[1].set_ylabel("Range sample [pixel]")
    axes[1].set_title("Stage 3 — Subsection #0: peak-elevation intensity map")
    figure.colorbar(intensity_image, ax=axes[1], label="Peak intensity [linear]")

    figure.savefig(figure_directory / "stage_3_dispatch_workers.pdf")
    plt.show()
else:
    print("No HDF5 partials present — PyRat focusing did not run in this environment.")

In [ ]:
distinct_suffixes      = {pyrat_job.suffix for pyrat_job in reduced_pyrat_jobs}
shared_method          = len({pyrat_job.beamforming_method for pyrat_job in reduced_pyrat_jobs}) == 1
shared_height_range    = len({tuple(pyrat_job.height_range) for pyrat_job in reduced_pyrat_jobs}) == 1
shared_stack           = len({pyrat_job.stack_identifier for pyrat_job in reduced_pyrat_jobs}) == 1
partial_datasets_ok    = True
for partial_file_path in partial_file_paths:
    with h5py.File(str(partial_file_path), "r") as hdf5_file:
        partial_datasets_ok = partial_datasets_ok and ("tomogram" in hdf5_file) and ("DEM" in hdf5_file)

stage_3_checks = [
    ("One job per Stage-2 subsection",            len(reduced_pyrat_jobs) == len(reduced_subsection_crops)),
    ("All job suffixes are distinct",             len(distinct_suffixes) == len(reduced_pyrat_jobs)),
    ("All jobs share beamforming method",         shared_method),
    ("All jobs share height range",               shared_height_range),
    ("All jobs share stack identifier",           shared_stack),
    ("Worker count never exceeds job count",      resolved_worker_count <= len(reduced_pyrat_jobs)),
    ("Worker count is at least one",              resolved_worker_count >= 1),
    ("One HDF5 partial written per job",          len(partial_file_paths) == len(reduced_pyrat_jobs)),
    ("Every partial exposes tomogram and DEM",    partial_datasets_ok),
]
for stage_3_description, stage_3_condition in stage_3_checks:
    print(f"[{'PASS' if stage_3_condition else 'FAIL'}] {stage_3_description}")

### Common Mistakes — Stage 3: Dispatch Parallel PyRat Workers

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Partials overwrite each other; only one survives | duplicate `suffix` across jobs (loop index not used) | Print `distinct_suffixes`; it must have one entry per job |
| Focused intensity is meaningless / inverted | baseline / acquisition ordering mismatch between the steering vector and the stack columns | Verify the stack id and `track_selection` are identical across jobs and match the SLC load order |
| Tomogram has only magnitude, phase lost | reading `tomogram` as real instead of complex when forming the steering covariance | Check `hdf5_file['tomogram'].dtype` is complex; intensity is `abs`, not `real` |
| Workers deadlock or thrash the CPU | `pyrat_threads × workers` exceeds available cores | Compare `resolved_worker_count × pyrat_threads` with `available_cores()` |
| Non-deterministic partial contents between runs | a worker reads global state mutated by `spawn` not re-initialising PyRat per process | Confirm `run_pyrat` calls `pyrat_init` inside the child and `parent_sys_path` is reset |
| Height axis flipped (canopy at the bottom) | `height_range` passed as `(high, low)` or steering vector sign convention reversed | Inspect the elevation profile orientation against `height_range` |
| `LD_LIBRARY_PATH` / Qt errors crash workers | conda lib path or offscreen platform not set in the child | The worker sets `QT_QPA_PLATFORM=offscreen` and prepends `sys.prefix/lib` |

**Passing to Stage 4:** `inspection_temporary_directory` — `Path`, the directory of per-subsection HDF5 partials that the concatenator merges along azimuth into a single tomogram cube and DEM.

---
## Stage 4: Concatenate Subsection Partials

**Callable:** `TomogramProcessor._concatenate(temporary_directory)`
**Input:** the temporary directory holding the $M$ HDF5 partials from Stage 3.
**Output:** `(combined_dem, combined_tomogram)` — two NumPy arrays. The DEM is stacked along axis 0 (azimuth), the tomogram along axis 1 (azimuth), recovering the original full crop.

With per-subsection tomograms $T^{(i)}$ of shape $(H, A_i, R)$ (elevation $\times$ azimuth $\times$ range) and DEMs $D^{(i)}$ of shape $(A_i, R)$, concatenation forms

$$T = \big[\,T^{(0)} \;\Vert_{\text{az}}\; T^{(1)} \;\Vert_{\text{az}}\; \dots \;\big], \quad A = \sum_i A_i, \qquad D = \big[\,D^{(0)} \;\Vert_{\text{az}}\; D^{(1)} \;\dots\big]$$

The partials are read in **sorted filename order**, so the suffix ordering from Stage 3 (`0000`, `0001`, …) must match the azimuth tiling order, otherwise azimuth columns are stitched out of sequence.

> **What you should see:** a `combined_tomogram` whose elevation (axis 0) and range (axis 2) extents equal those of any single partial, and whose azimuth extent (axis 1) equals the sum of the per-partial azimuth widths — i.e. the original crop azimuth size. A `combined_dem` whose axis-0 length equals that same azimuth size. Both arrays inherit the partials' dtype (the tomogram complex, the DEM real). No NaN or Inf in either.

In [ ]:
combined_dem, combined_tomogram = tomogram_processor._concatenate(inspection_temporary_directory)

In [ ]:
combined_tomogram_intensity = np.abs(combined_tomogram).astype(np.float64)

print("combined_tomogram shape  :", combined_tomogram.shape, "(elevation, azimuth, range)")
print("combined_tomogram dtype  :", combined_tomogram.dtype)
print("combined_dem shape       :", combined_dem.shape, "(azimuth, range)")
print("combined_dem dtype       :", combined_dem.dtype)
print()
print("Tomogram is complex      :", np.iscomplexobj(combined_tomogram))
print("Intensity min / max      :", float(combined_tomogram_intensity.min()), float(combined_tomogram_intensity.max()))
print("Intensity mean / std     :", float(combined_tomogram_intensity.mean()), float(combined_tomogram_intensity.std()))
print("DEM min / max [m]        :", float(np.nanmin(combined_dem)), float(np.nanmax(combined_dem)))
print()
print("Sum of partial azimuths  :", sum(subsection_widths.tolist()))
print("Tomogram azimuth extent  :", combined_tomogram.shape[1])
print("DEM azimuth extent       :", combined_dem.shape[0])
print()
print("NaN in tomogram          :", bool(np.isnan(combined_tomogram_intensity).any()))
print("Inf in tomogram          :", bool(np.isinf(combined_tomogram_intensity).any()))
print("NaN in DEM               :", bool(np.isnan(np.asarray(combined_dem, dtype=np.float64)).any()))

In [ ]:
elevation_count = combined_tomogram.shape[0]
height_axis     = np.linspace(resolved_reduced_variant.tomogram_config.height_range[0], resolved_reduced_variant.tomogram_config.height_range[1], elevation_count)

mid_range_index    = combined_tomogram.shape[2] // 2
elevation_azimuth  = combined_tomogram_intensity[:, :, mid_range_index]
peak_height_index  = combined_tomogram_intensity.argmax(axis=0)
peak_height_map    = height_axis[peak_height_index]

figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
elevation_image = axes[0].imshow(
    elevation_azimuth,
    aspect="auto",
    cmap=intensity_colormap,
    origin="lower",
    extent=[0, combined_tomogram.shape[1], height_axis[0], height_axis[-1]],
)
axes[0].set_xlabel("Azimuth sample [pixel]")
axes[0].set_ylabel("Elevation / height [m]")
axes[0].set_title(f"Stage 4 — Tomogram intensity along elevation (range bin {mid_range_index})")
figure.colorbar(elevation_image, ax=axes[0], label="Intensity [linear]")

height_image = axes[1].imshow(peak_height_map.T, aspect="auto", cmap=height_colormap, origin="lower")
axes[1].set_xlabel("Azimuth sample [pixel]")
axes[1].set_ylabel("Range sample [pixel]")
axes[1].set_title("Stage 4 — Peak-scatterer height map")
figure.colorbar(height_image, ax=axes[1], label="Height of intensity peak [m]")

figure.savefig(figure_directory / "stage_4_concatenate.pdf")
plt.show()

In [ ]:
with h5py.File(str(partial_file_paths[0]), "r") as hdf5_file:
    single_partial_elevation = hdf5_file["tomogram"].shape[0]
    single_partial_range     = hdf5_file["tomogram"].shape[2]

expected_combined_azimuth = int(sum(subsection_widths.tolist()))

stage_4_checks = [
    ("Tomogram is a 3-D array",                     combined_tomogram.ndim == 3),
    ("DEM is a 2-D array",                          combined_dem.ndim == 2),
    ("Tomogram elevation matches a partial",        combined_tomogram.shape[0] == single_partial_elevation),
    ("Tomogram range matches a partial",            combined_tomogram.shape[2] == single_partial_range),
    ("Tomogram azimuth equals sum of widths",       combined_tomogram.shape[1] == expected_combined_azimuth),
    ("DEM azimuth equals sum of widths",            combined_dem.shape[0] == expected_combined_azimuth),
    ("Tomogram dtype is complex",                   np.iscomplexobj(combined_tomogram)),
    ("No NaN in tomogram intensity",                not bool(np.isnan(combined_tomogram_intensity).any())),
    ("No Inf in tomogram intensity",                not bool(np.isinf(combined_tomogram_intensity).any())),
    ("Tomogram intensity is non-negative",          float(combined_tomogram_intensity.min()) >= 0.0),
]
for stage_4_description, stage_4_condition in stage_4_checks:
    print(f"[{'PASS' if stage_4_condition else 'FAIL'}] {stage_4_description}")

### Common Mistakes — Stage 4: Concatenate Subsection Partials

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Azimuth columns appear shuffled / discontinuous | partials concatenated in OS directory order, not sorted suffix order | Confirm `sorted(partial_files_directory.iterdir())` and that suffixes increase with azimuth start |
| Combined azimuth extent wrong by one tile | a partial missing because a worker failed silently | Compare `len(partial_file_paths)` with the Stage-2 subsection count |
| DEM and tomogram azimuth disagree | tomogram concatenated on axis 1 but DEM on the wrong axis | Check `combined_dem.shape[0] == combined_tomogram.shape[1]` |
| Elevation or range extent changes after merge | a partial focused with a different `height_range` or crop range | Verify all partials share elevation (axis 0) and range (axis 2) before allocation |
| Intensity has NaN streaks at tile seams | covariance singular at a window straddling the seam | Inspect the peak-height map for vertical artefacts at subsection boundaries |
| Complex tomogram silently cast to real | `np.empty(..., dtype=tomogram_dtype)` read from a real-typed partial | Assert `np.iscomplexobj(combined_tomogram)` |

**Passing to Stage 5:** `combined_tomogram` and `combined_dem` are the in-memory products that `_stage_tomogram` saves to disk; the next stage runs the full end-to-end variant call that wraps Stages 2–4 and persists the result with metadata.

---
## Stage 5: Focus and Persist the FULL Tomogram

**Callable:** `ProcessingPipeline._stage_tomogram("full")` (wraps `TomogramProcessor.run()` + `MetadataManager`)
**Input:** the variant literal `"full"`.
**Output:** `(tomogram_full_path, dem_full_path)` — the saved `.npy` artifacts — plus a metadata text file. This is the first heavy call `run()` issues.

This stage is the orchestrated composition of Stages 2–4 followed by persistence: resolve the **full / parameter** variant, subdivide the crop, dispatch PyRat workers, concatenate, save to `tomogram_full_<parameter_tag>.npy` and `dem_full_<parameter_tag>.npy`, then write `meta_tomogram_full_<parameter_tag>.txt`. The full variant uses `output_config` (the parameter-side `TomogramConfiguration`) and the `full_stack_identifier`.

> **What you should see:** two `.npy` files on disk under the data directory whose names embed the **parameter** tag (not the tomogram tag). Reloading the tomogram yields a 3-D complex cube (elevation $\times$ azimuth $\times$ range) whose azimuth and range match the global crop, and a 2-D real DEM. A metadata file `meta_tomogram_full_<parameter_tag>.txt` whose `crop`, `id`, `range` and `method` entries echo the full-variant config.

In [ ]:
inspection_pipeline = ProcessingPipeline.__new__(ProcessingPipeline)
inspection_pipeline.config             = processing_configuration
inspection_pipeline.logger             = processing_logger
inspection_pipeline.artifact_registry  = artifact_registry
inspection_pipeline.metadata_manager   = metadata_manager
inspection_pipeline.tomogram_processor = tomogram_processor
inspection_pipeline.interferogram_builder = interferogram_builder

full_tomogram_path, full_dem_path = inspection_pipeline._stage_tomogram("full")

In [ ]:
full_tomogram_array = np.load(str(full_tomogram_path), allow_pickle=False)
full_dem_array      = np.load(str(full_dem_path),      allow_pickle=False)
full_tomogram_intensity = np.abs(full_tomogram_array).astype(np.float64)

full_metadata_path = processing_configuration.paths.metadata_directory / f"meta_tomogram_full_{processing_configuration.parameter_tag}.txt"
full_metadata_text = full_metadata_path.read_text(encoding="utf-8") if full_metadata_path.exists() else ""

print("Tomogram path            :", full_tomogram_path)
print("DEM path                 :", full_dem_path)
print("Tomogram file name        :", full_tomogram_path.name)
print()
print("Tomogram shape           :", full_tomogram_array.shape, "(elevation, azimuth, range)")
print("Tomogram dtype           :", full_tomogram_array.dtype)
print("DEM shape                :", full_dem_array.shape)
print("DEM dtype                :", full_dem_array.dtype)
print()
print("Intensity min/mean/max   :", float(full_tomogram_intensity.min()), float(full_tomogram_intensity.mean()), float(full_tomogram_intensity.max()))
print("DEM min/max [m]          :", float(np.nanmin(full_dem_array)), float(np.nanmax(full_dem_array)))
print()
print("Metadata file exists     :", full_metadata_path.exists())
print("--- metadata ---")
print(full_metadata_text)

In [ ]:
full_elevation_count = full_tomogram_array.shape[0]
full_height_axis     = np.linspace(processing_configuration.output_config.height_range[0], processing_configuration.output_config.height_range[1], full_elevation_count)
full_mean_profile    = full_tomogram_intensity.reshape(full_elevation_count, -1).mean(axis=1)

figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(full_mean_profile, full_height_axis, color="#7a1f3d", linewidth=1.5)
axes[0].set_xlabel("Mean focused intensity [linear]")
axes[0].set_ylabel("Elevation / height [m]")
axes[0].set_title("Stage 5 — FULL tomogram: scene-averaged elevation profile")

full_dem_image = axes[1].imshow(full_dem_array.T, aspect="auto", cmap="terrain", origin="lower")
axes[1].set_xlabel("Azimuth sample [pixel]")
axes[1].set_ylabel("Range sample [pixel]")
axes[1].set_title("Stage 5 — FULL DEM (terrain reference)")
figure.colorbar(full_dem_image, ax=axes[1], label="Ground height [m]")

figure.savefig(figure_directory / "stage_5_full_tomogram.pdf")
plt.show()

In [ ]:
stage_5_checks = [
    ("Tomogram .npy exists",                       full_tomogram_path.is_file()),
    ("DEM .npy exists",                            full_dem_path.is_file()),
    ("Tomogram name carries parameter_tag",        processing_configuration.parameter_tag in full_tomogram_path.name),
    ("DEM name carries parameter_tag",             processing_configuration.parameter_tag in full_dem_path.name),
    ("Tomogram is 3-D complex",                    full_tomogram_array.ndim == 3 and np.iscomplexobj(full_tomogram_array)),
    ("DEM is 2-D real",                            full_dem_array.ndim == 2 and not np.iscomplexobj(full_dem_array)),
    ("Tomogram azimuth equals crop azimuth size",  full_tomogram_array.shape[1] == processing_configuration.crop.azimuth_size),
    ("DEM azimuth equals crop azimuth size",       full_dem_array.shape[0] == processing_configuration.crop.azimuth_size),
    ("No NaN in tomogram intensity",               not bool(np.isnan(full_tomogram_intensity).any())),
    ("Metadata file written",                      full_metadata_path.exists()),
    ("Metadata embeds full stack id",              processing_configuration.full_stack_identifier in full_metadata_text),
]
for stage_5_description, stage_5_condition in stage_5_checks:
    print(f"[{'PASS' if stage_5_condition else 'FAIL'}] {stage_5_description}")

### Common Mistakes — Stage 5: Focus and Persist the FULL Tomogram

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Full tomogram saved with the reduced tag | `_resolve_variant("full")` returns `tomogram_tag` instead of `parameter_tag` | Assert `parameter_tag in full_tomogram_path.name` |
| Full and reduced products identical | `output_config` falls back to `input_configs` and stack ids coincide | Compare full vs reduced stack ids and configs |
| Azimuth extent short by the last tile | a worker subsection silently dropped before concatenation | Check `full_tomogram_array.shape[1] == crop.azimuth_size` |
| Metadata `range` / `win` empty | `filter_arguments['win']` missing on the full config | Inspect the metadata text for non-empty `range` and `win` |
| DEM looks like noise, not terrain | DEM and tomogram datasets swapped on save | Plot the DEM; it should be a smooth terrain surface in metres |
| `.npy` reload fails / pickled object | array saved with `allow_pickle=True` and a non-array dtype | The save uses `allow_pickle=False`; confirm the dtype is numeric |

**Passing to Stage 6:** `full_tomogram_path`, `full_dem_path` — `Path`s to the persisted full / parameter products. `run()` next focuses the *reduced* variant by the same machinery.

---
## Stage 6: Focus and Persist the REDUCED Tomogram

**Callable:** `ProcessingPipeline._stage_tomogram("reduced")`
**Input:** the variant literal `"reduced"`.
**Output:** `(tomogram_reduced_path, dem_reduced_path)` — the saved `.npy` artifacts — plus `meta_tomogram_reduced_<tomogram_tag>.txt`. This is the second heavy call `run()` issues.

Identical machinery to Stage 5 but keyed on the **reduced** acquisition: `reduced_stack_identifier` and the `input_configs` `TomogramConfiguration`, persisted under the **tomogram** tag. The reduced stack has fewer acquisitions (a thinner baseline aperture), so its elevation resolution

$$\delta h \;\propto\; \frac{\lambda \, r}{2 \, B_{\perp}}$$

is coarser than the full stack's, where $B_{\perp}$ is the perpendicular baseline span and $r$ the slant range. The reduced tomogram is the model input; the full tomogram supplies the parameter target.

> **What you should see:** two `.npy` files embedding the **tomogram** tag, a 3-D complex reduced cube and a 2-D real DEM, both with the same azimuth/range extent as the full variant (same crop) but typically a coarser / different elevation sampling. A metadata file echoing the reduced-variant config, with `id == reduced_stack_identifier`.

In [ ]:
reduced_tomogram_path, reduced_dem_path = inspection_pipeline._stage_tomogram("reduced")

In [ ]:
reduced_tomogram_array = np.load(str(reduced_tomogram_path), allow_pickle=False)
reduced_dem_array      = np.load(str(reduced_dem_path),      allow_pickle=False)
reduced_tomogram_intensity = np.abs(reduced_tomogram_array).astype(np.float64)

reduced_metadata_path = processing_configuration.paths.metadata_directory / f"meta_tomogram_reduced_{processing_configuration.tomogram_tag}.txt"
reduced_metadata_text = reduced_metadata_path.read_text(encoding="utf-8") if reduced_metadata_path.exists() else ""

print("Reduced tomogram path    :", reduced_tomogram_path)
print("Reduced DEM path         :", reduced_dem_path)
print("Reduced tomogram name     :", reduced_tomogram_path.name)
print()
print("Reduced tomogram shape   :", reduced_tomogram_array.shape, "(elevation, azimuth, range)")
print("Reduced tomogram dtype   :", reduced_tomogram_array.dtype)
print("Reduced DEM shape        :", reduced_dem_array.shape)
print()
print("FULL elevation samples   :", full_tomogram_array.shape[0])
print("REDUCED elevation samples:", reduced_tomogram_array.shape[0])
print("Reduced intensity min/mean/max:", float(reduced_tomogram_intensity.min()), float(reduced_tomogram_intensity.mean()), float(reduced_tomogram_intensity.max()))
print()
print("Metadata file exists     :", reduced_metadata_path.exists())
print("--- metadata ---")
print(reduced_metadata_text)

In [ ]:
reduced_elevation_count = reduced_tomogram_array.shape[0]
reduced_height_axis     = np.linspace(processing_configuration.input_configs.height_range[0], processing_configuration.input_configs.height_range[1], reduced_elevation_count)
reduced_mean_profile    = reduced_tomogram_intensity.reshape(reduced_elevation_count, -1).mean(axis=1)

full_mean_profile_norm    = full_mean_profile / (full_mean_profile.max() + 1e-30)
reduced_mean_profile_norm = reduced_mean_profile / (reduced_mean_profile.max() + 1e-30)

figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
axes[0].plot(full_mean_profile_norm,    full_height_axis,    color="#7a1f3d", linewidth=1.5, label="full (parameter)")
axes[0].plot(reduced_mean_profile_norm, reduced_height_axis, color="#1b3b6f", linewidth=1.5, label="reduced (input)")
axes[0].set_xlabel("Normalised mean intensity [-]")
axes[0].set_ylabel("Elevation / height [m]")
axes[0].set_title("Stage 6 — Elevation profile: full vs reduced aperture")
axes[0].legend(frameon=False, loc="best")

reduced_mid_range = reduced_tomogram_array.shape[2] // 2
reduced_slice     = reduced_tomogram_intensity[:, :, reduced_mid_range]
reduced_image = axes[1].imshow(
    reduced_slice,
    aspect="auto",
    cmap=intensity_colormap,
    origin="lower",
    extent=[0, reduced_tomogram_array.shape[1], reduced_height_axis[0], reduced_height_axis[-1]],
)
axes[1].set_xlabel("Azimuth sample [pixel]")
axes[1].set_ylabel("Elevation / height [m]")
axes[1].set_title(f"Stage 6 — REDUCED tomogram along elevation (range bin {reduced_mid_range})")
figure.colorbar(reduced_image, ax=axes[1], label="Intensity [linear]")

figure.savefig(figure_directory / "stage_6_reduced_tomogram.pdf")
plt.show()

In [ ]:
stage_6_checks = [
    ("Reduced tomogram .npy exists",               reduced_tomogram_path.is_file()),
    ("Reduced DEM .npy exists",                    reduced_dem_path.is_file()),
    ("Tomogram name carries tomogram_tag",         processing_configuration.tomogram_tag in reduced_tomogram_path.name),
    ("Reduced tomogram is 3-D complex",            reduced_tomogram_array.ndim == 3 and np.iscomplexobj(reduced_tomogram_array)),
    ("Reduced DEM is 2-D real",                    reduced_dem_array.ndim == 2 and not np.iscomplexobj(reduced_dem_array)),
    ("Reduced azimuth equals crop azimuth size",   reduced_tomogram_array.shape[1] == processing_configuration.crop.azimuth_size),
    ("Reduced range equals full range",            reduced_tomogram_array.shape[2] == full_tomogram_array.shape[2]),
    ("Reduced DEM azimuth matches tomogram",       reduced_dem_array.shape[0] == reduced_tomogram_array.shape[1]),
    ("No NaN in reduced intensity",                not bool(np.isnan(reduced_tomogram_intensity).any())),
    ("Metadata embeds reduced stack id",           processing_configuration.reduced_stack_identifier in reduced_metadata_text),
]
for stage_6_description, stage_6_condition in stage_6_checks:
    print(f"[{'PASS' if stage_6_condition else 'FAIL'}] {stage_6_description}")

### Common Mistakes — Stage 6: Focus and Persist the REDUCED Tomogram

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Reduced and full tomograms have identical elevation sampling | reduced variant focused with the full stack / config | Compare elevation sample counts and stack ids across the two products |
| Reduced product saved under the parameter tag | tag swap in `_resolve_variant` | Assert `tomogram_tag in reduced_tomogram_path.name` |
| Reduced and full ranges disagree | different range crop on one variant | Check `reduced.shape[2] == full.shape[2]` |
| Coarse reduced elevation interpreted as a bug | thinner baseline aperture genuinely lowers $\delta h$ resolution | Confirm against the baseline span; coarser sampling is expected, not an error |
| Spatial misalignment between reduced tomogram and full DEM | the two variants focused on shifted crops | Confirm both share `crop.azimuth_size` and `range_size` |
| Reduced DEM differs from reduced tomogram azimuth | DEM concatenation axis mismatch | Assert `reduced_dem.shape[0] == reduced_tomogram.shape[1]` |

**Passing to Stage 7:** the reduced acquisition is now focused; `run()` next forms the interferometric stack for that **same reduced acquisition** from its SLCs.

---
## Stage 7: Form the Interferometric Stack

**Callable:** `InterferogramBuilder.build(crop_tuple)` (delegates to `_build_from_fsar` → `_compute_interferograms`)
**Input:** the global crop tuple `config.crop.as_tuple()`.
**Output:** `(primary, secondaries, interferograms)` — three complex64 arrays. `primary` is the master SLC of shape $(R, A)$; `secondaries` and `interferograms` are stacks of shape $(N_{\text{sec}}, R, A)$.

For each secondary acquisition $s$, the builder loads the secondary SLC, removes the DEM / flat-earth phase ramp $\phi^{\text{DEM}}_s$, forms the interferometric phasor against the primary $p$, normalises it to unit magnitude, and re-weights by the clipped secondary amplitude:

$$\tilde s = s\,e^{\,i\phi^{\text{DEM}}_s}, \qquad z = \frac{p\,\tilde s^{*}}{|p\,\tilde s^{*}| + 10^{-30}}, \qquad I_s = \min(|s|,\,c_{\max})\;\cdot\;z$$

so $\angle I_s$ is the DEM-deramped interferometric phase and $|I_s| = \min(|s|, c_{\max})$ is the clipped secondary amplitude ($c_{\max} = $ `max_amplitude_clip`). The interferogram is therefore an **amplitude-weighted unit phasor**, not a raw cross-product.

> **What you should see:** a `primary` of shape $(R, A)$ matching the crop range/azimuth, `complex64`. `secondaries` and `interferograms` both of shape $(N_{\text{sec}}, R, A)$, `complex64`, with $N_{\text{sec}} = $ number of slaves. Every interferogram magnitude lies in $[0, c_{\max}]$ (clipped at `max_amplitude_clip = 1.25`). Phase $\angle I_s \in (-\pi, \pi]$, wrapped. No NaN / Inf.

In [ ]:
interferogram_primary, interferogram_secondaries, interferogram_stack = interferogram_builder.build(
    processing_configuration.crop.as_tuple()
)

In [ ]:
primary_amplitude        = np.abs(interferogram_primary).astype(np.float64)
interferogram_amplitude  = np.abs(interferogram_stack).astype(np.float64)
interferogram_phase       = np.angle(interferogram_stack).astype(np.float64)
amplitude_clip_limit     = processing_configuration.input_configs.max_amplitude_clip

print("primary shape            :", interferogram_primary.shape, "(range, azimuth)")
print("primary dtype            :", interferogram_primary.dtype)
print("secondaries shape        :", interferogram_secondaries.shape, "(n_secondaries, range, azimuth)")
print("secondaries dtype        :", interferogram_secondaries.dtype)
print("interferograms shape     :", interferogram_stack.shape, "(n_secondaries, range, azimuth)")
print("interferograms dtype     :", interferogram_stack.dtype)
print()
print("Number of secondaries    :", interferogram_secondaries.shape[0])
print("Amplitude clip limit     :", amplitude_clip_limit)
print()
print("primary amplitude min/max:", float(primary_amplitude.min()), float(primary_amplitude.max()))
print("interferogram |I| min/max:", float(interferogram_amplitude.min()), float(interferogram_amplitude.max()))
print("interferogram phase range:", float(interferogram_phase.min()), float(interferogram_phase.max()))
print()
print("NaN in interferograms    :", bool(np.isnan(interferogram_amplitude).any()))
print("Inf in interferograms    :", bool(np.isinf(interferogram_amplitude).any()))
print("All complex64            :", interferogram_primary.dtype == np.complex64 and interferogram_stack.dtype == np.complex64)

In [ ]:
first_interferogram_phase     = interferogram_phase[0]
first_interferogram_amplitude = interferogram_amplitude[0]

figure, axes = plt.subplots(1, 3, figsize=(15.0, 4.2))

phase_image = axes[0].imshow(first_interferogram_phase, aspect="auto", cmap=phase_colormap, origin="lower", vmin=-np.pi, vmax=np.pi)
axes[0].set_xlabel("Azimuth sample [pixel]")
axes[0].set_ylabel("Range sample [pixel]")
axes[0].set_title("Stage 7 — Interferogram #0: DEM-deramped phase")
figure.colorbar(phase_image, ax=axes[0], label="Phase [rad]")

amplitude_image = axes[1].imshow(first_interferogram_amplitude, aspect="auto", cmap=intensity_colormap, origin="lower", vmin=0.0, vmax=amplitude_clip_limit)
axes[1].set_xlabel("Azimuth sample [pixel]")
axes[1].set_ylabel("Range sample [pixel]")
axes[1].set_title("Stage 7 — Interferogram #0: clipped amplitude")
figure.colorbar(amplitude_image, ax=axes[1], label="Amplitude [linear]")

axes[2].hist(first_interferogram_phase.ravel(), bins=120, color="#3b6f4f", edgecolor="none")
axes[2].axvline(0.0, color="black", linewidth=1.0, linestyle="--")
axes[2].set_xlabel("Phase [rad]")
axes[2].set_ylabel("Pixel count [-]")
axes[2].set_title("Stage 7 — Interferogram #0: phase distribution")
axes[2].set_xlim(-np.pi, np.pi)

figure.savefig(figure_directory / "stage_7_interferograms.pdf")
plt.show()

In [ ]:
number_of_secondaries = interferogram_secondaries.shape[0]
spatial_shapes_match  = (interferogram_primary.shape == interferogram_secondaries.shape[1:]) and (interferogram_primary.shape == interferogram_stack.shape[1:])

stage_7_checks = [
    ("primary is 2-D complex64",                   interferogram_primary.ndim == 2 and interferogram_primary.dtype == np.complex64),
    ("secondaries is 3-D complex64",               interferogram_secondaries.ndim == 3 and interferogram_secondaries.dtype == np.complex64),
    ("interferograms is 3-D complex64",            interferogram_stack.ndim == 3 and interferogram_stack.dtype == np.complex64),
    ("secondaries and interferograms same count",  interferogram_secondaries.shape[0] == interferogram_stack.shape[0]),
    ("primary spatial shape matches stacks",       spatial_shapes_match),
    ("interferogram magnitude >= 0",               float(interferogram_amplitude.min()) >= 0.0),
    ("interferogram magnitude <= clip limit",      float(interferogram_amplitude.max()) <= amplitude_clip_limit + 1e-5),
    ("phase within (-pi, pi]",                     float(interferogram_phase.min()) >= -np.pi - 1e-5 and float(interferogram_phase.max()) <= np.pi + 1e-5),
    ("No NaN in interferograms",                   not bool(np.isnan(interferogram_amplitude).any())),
    ("At least one secondary present",             number_of_secondaries >= 1),
]
for stage_7_description, stage_7_condition in stage_7_checks:
    print(f"[{'PASS' if stage_7_condition else 'FAIL'}] {stage_7_description}")

### Common Mistakes — Stage 7: Form the Interferometric Stack

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Interferogram is a raw magnitude, phase lost | model trained on `np.abs(...)` instead of the complex phasor | Confirm dtype is `complex64` and `np.angle` is non-trivial |
| Strong residual fringes everywhere | DEM phase not removed — `dem_phase` sign wrong or skipped | The phasor uses `np.exp(+1j*dem_phase)`; inspect the phase map for flat-earth ramps |
| Phase appears unwrapped / outside $(-\pi,\pi]$ | wrapping convention broken, or phase taken before normalisation | Plot the phase histogram; it must lie within $\pm\pi$ |
| Amplitudes saturate uniformly at the clip | `max_amplitude_clip` set too low for the scene's backscatter | Compare `interferogram |I| max` against `max_amplitude_clip` |
| Primary / secondary swapped (conjugate flips) | `np.conj` applied to the wrong operand | Phase sign inverts; verify `phasor = primary * conj(deramped_secondary)` |
| `n_secondaries` off by one | master accidentally included in the slaves list | Check `number_of_secondaries == len(tomography_object.slaves)` |
| Spatial shapes disagree between primary and stack | crop applied inconsistently across SLC loads | Assert `primary.shape == interferograms.shape[1:]` |

**Passing to Stage 8:** `interferogram_primary`, `interferogram_secondaries`, `interferogram_stack` — the in-memory complex products that `_stage_inputs()` persists to `.npy` with an inputs-metadata record.

---
## Stage 8: Persist the Interferometric Inputs

**Callable:** `ProcessingPipeline._stage_inputs()` (delegates to `InterferogramBuilder.run()` + `MetadataManager.build_inputs_metadata`)
**Input:** the global crop tuple.
**Output:** `(primary_path, secondaries_path, interferograms_path)` — three `.npy` artifacts under the data directory — plus `meta_inputs_<tomogram_tag>.txt`. This is the third and final heavy call `run()` issues before the layout manifest.

No new signal transformation: `run()` rebuilds the stack (Stage 7) and writes each array to disk, returning the **shapes** (not the arrays) for the metadata record. The three filenames embed the **tomogram** tag so they collate with the reduced tomogram of Stage 6.

> **What you should see:** three `.npy` files on disk whose names embed the tomogram tag. Reloaded, the primary is 2-D complex64 and the secondaries / interferograms are 3-D complex64 with matching spatial shape — identical to the in-memory Stage-7 products. A metadata file whose `primary_shape`, `secondaries_shape`, `interferograms_shape` strings equal the saved array shapes and whose `id == reduced_stack_identifier`.

In [ ]:
primary_artifact_path        = artifact_registry.artifact_path("primary_reduced")
secondaries_artifact_path    = artifact_registry.artifact_path("secondaries_reduced")
interferograms_artifact_path = artifact_registry.artifact_path("interferograms_reduced")

primary_shape, secondaries_shape, interferograms_shape = interferogram_builder.run(
    crop_tuple          = processing_configuration.crop.as_tuple(),
    primary_path        = primary_artifact_path,
    secondaries_path    = secondaries_artifact_path,
    interferograms_path = interferograms_artifact_path,
)

inputs_metadata_path = metadata_manager.save_stage_metadata(
    stage_name       = "inputs",
    identifier_tag   = processing_configuration.tomogram_tag,
    metadata_entries = metadata_manager.build_inputs_metadata(
        primary_artifact_path, secondaries_artifact_path, interferograms_artifact_path,
        primary_shape, secondaries_shape, interferograms_shape,
    ),
)

In [ ]:
reloaded_primary        = np.load(str(primary_artifact_path),        allow_pickle=False)
reloaded_secondaries    = np.load(str(secondaries_artifact_path),    allow_pickle=False)
reloaded_interferograms = np.load(str(interferograms_artifact_path), allow_pickle=False)
inputs_metadata_text    = inputs_metadata_path.read_text(encoding="utf-8")

print("primary path             :", primary_artifact_path)
print("secondaries path         :", secondaries_artifact_path)
print("interferograms path      :", interferograms_artifact_path)
print()
print("Returned primary shape   :", primary_shape)
print("Returned secondaries shape:", secondaries_shape)
print("Returned interfero shape :", interferograms_shape)
print()
print("Reloaded primary         :", reloaded_primary.shape, reloaded_primary.dtype)
print("Reloaded secondaries     :", reloaded_secondaries.shape, reloaded_secondaries.dtype)
print("Reloaded interferograms  :", reloaded_interferograms.shape, reloaded_interferograms.dtype)
print()
print("Metadata file exists     :", inputs_metadata_path.exists())
print("--- metadata ---")
print(inputs_metadata_text)

In [ ]:
reloaded_interferogram_amplitude = np.abs(reloaded_interferograms).astype(np.float64)
per_secondary_mean_amplitude     = reloaded_interferogram_amplitude.reshape(reloaded_interferograms.shape[0], -1).mean(axis=1)
per_secondary_mean_phase         = np.angle(reloaded_interferograms).reshape(reloaded_interferograms.shape[0], -1).mean(axis=1)

figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
secondary_indices = np.arange(reloaded_interferograms.shape[0])

axes[0].bar(secondary_indices, per_secondary_mean_amplitude, color="#1b3b6f", edgecolor="black", linewidth=0.6)
axes[0].set_xlabel("Secondary acquisition index [-]")
axes[0].set_ylabel("Mean interferogram amplitude [linear]")
axes[0].set_title("Stage 8 — Per-secondary mean amplitude")

axes[1].bar(secondary_indices, per_secondary_mean_phase, color="#7a1f3d", edgecolor="black", linewidth=0.6)
axes[1].axhline(0.0, color="black", linewidth=1.0, linestyle="--")
axes[1].set_xlabel("Secondary acquisition index [-]")
axes[1].set_ylabel("Mean interferogram phase [rad]")
axes[1].set_title("Stage 8 — Per-secondary mean phase (baseline trend)")

figure.savefig(figure_directory / "stage_8_inputs_persist.pdf")
plt.show()

In [ ]:
stage_8_checks = [
    ("primary .npy exists",                        primary_artifact_path.is_file()),
    ("secondaries .npy exists",                    secondaries_artifact_path.is_file()),
    ("interferograms .npy exists",                 interferograms_artifact_path.is_file()),
    ("All names carry tomogram_tag",               all(processing_configuration.tomogram_tag in p.name for p in (primary_artifact_path, secondaries_artifact_path, interferograms_artifact_path))),
    ("Reloaded primary is complex64 2-D",          reloaded_primary.dtype == np.complex64 and reloaded_primary.ndim == 2),
    ("Reloaded interferograms complex64 3-D",      reloaded_interferograms.dtype == np.complex64 and reloaded_interferograms.ndim == 3),
    ("Returned shape equals reloaded primary",     tuple(primary_shape) == reloaded_primary.shape),
    ("Returned shape equals reloaded interfero",   tuple(interferograms_shape) == reloaded_interferograms.shape),
    ("secondaries / interferograms same count",    reloaded_secondaries.shape[0] == reloaded_interferograms.shape[0]),
    ("Metadata embeds reduced stack id",           processing_configuration.reduced_stack_identifier in inputs_metadata_text),
    ("Metadata file written",                      inputs_metadata_path.exists()),
]
for stage_8_description, stage_8_condition in stage_8_checks:
    print(f"[{'PASS' if stage_8_condition else 'FAIL'}] {stage_8_description}")

### Common Mistakes — Stage 8: Persist the Interferometric Inputs

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Reloaded arrays are real, not complex | `.npy` round-trip through an intermediate real cast | Assert `reloaded_primary.dtype == np.complex64` |
| Metadata shapes disagree with saved arrays | `run()` returned in-memory shapes before a re-crop | Compare `interferograms_shape` against `reloaded_interferograms.shape` |
| Inputs collate with the wrong tomogram | filenames embed the parameter tag instead of the tomogram tag | Assert `tomogram_tag in` each artifact name |
| Per-secondary mean phase is flat (all zero) | DEM deramp already flattened genuine baseline phase, or phase wiped | Expect a baseline-dependent trend across secondary index |
| One secondary missing from the stack | a slave SLC failed to load and was skipped | Compare secondary count against the slaves list length |
| Huge `.npy` files / disk pressure | complex128 saved instead of complex64 | Confirm dtype is `complex64`; the builder casts explicitly |

**Passing to Stage 9:** all seven artifacts (two tomograms, two DEMs, primary, secondaries, interferograms) are now on disk; `run()` finally writes the manifest that catalogues them.

---
## Stage 9: Write the Dataset Layout Manifest

**Callable:** `MetadataManager.save_dataset_layout()`
**Input:** the live config and the artifact registry.
**Output:** a `Path` to `dataset.json` in the data directory, cataloguing the global crop, dataset type, tags, and the filename of every artifact.

No signal transformation. This manifest is the single source of truth a downstream dataset loader reads to locate the seven products without re-deriving the tag logic.

> **What you should see:** a `dataset.json` in the data directory whose `global_crop` equals `crop.as_tuple()`, whose `dataset_type` is `"FSAR"`, whose `tomogram_tag` / `parameter_tag` equal the live config, and whose `artifacts` mapping lists exactly the seven artifact filenames — each matching the names actually written in Stages 5–8.

In [ ]:
dataset_layout_path = metadata_manager.save_dataset_layout()
dataset_layout      = json.loads(Path(dataset_layout_path).read_text(encoding="utf-8"))

print("Dataset layout path      :", dataset_layout_path)
print("global_crop              :", dataset_layout["global_crop"])
print("dataset_type             :", dataset_layout["dataset_type"])
print("tomogram_tag             :", dataset_layout["tomogram_tag"])
print("parameter_tag            :", dataset_layout["parameter_tag"])
print()
print("Catalogued artifacts:")
for artifact_name, artifact_filename in dataset_layout["artifacts"].items():
    artifact_on_disk = (processing_configuration.paths.data_directory / artifact_filename).exists()
    print(f"  {artifact_name:<24}: {artifact_filename}  (exists={artifact_on_disk})")

In [ ]:
manifest_artifact_names    = list(dataset_layout["artifacts"].keys())
manifest_artifact_presence = [int((processing_configuration.paths.data_directory / dataset_layout["artifacts"][name]).exists()) for name in manifest_artifact_names]
manifest_artifact_sizes_mb = [(processing_configuration.paths.data_directory / dataset_layout["artifacts"][name]).stat().st_size / 1.0e6 if (processing_configuration.paths.data_directory / dataset_layout["artifacts"][name]).exists() else 0.0 for name in manifest_artifact_names]

figure, axis = plt.subplots(figsize=(9.0, 4.2))
bar_positions = np.arange(len(manifest_artifact_names))
bar_colors    = ["#1b3b6f" if present else "#9a9a9a" for present in manifest_artifact_presence]
axis.bar(bar_positions, manifest_artifact_sizes_mb, color=bar_colors, edgecolor="black", linewidth=0.6)
axis.set_xticks(bar_positions)
axis.set_xticklabels(manifest_artifact_names, rotation=35, ha="right")
axis.set_ylabel("On-disk size [MB]")
axis.set_title("Stage 9 — Dataset manifest: catalogued artifact sizes (grey = missing)")
figure.savefig(figure_directory / "stage_9_dataset_layout.pdf")
plt.show()

In [ ]:
catalogued_artifacts_present = all(
    (processing_configuration.paths.data_directory / filename).exists()
    for filename in dataset_layout["artifacts"].values()
)

stage_9_checks = [
    ("dataset.json exists",                        dataset_layout_path.is_file()),
    ("Manifest sits in data directory",            dataset_layout_path.parent == processing_configuration.paths.data_directory),
    ("global_crop equals live crop",               tuple(dataset_layout["global_crop"]) == processing_configuration.crop.as_tuple()),
    ("dataset_type is FSAR",                        dataset_layout["dataset_type"] == processing_configuration.dataset_type),
    ("tomogram_tag matches live config",           dataset_layout["tomogram_tag"] == processing_configuration.tomogram_tag),
    ("parameter_tag matches live config",          dataset_layout["parameter_tag"] == processing_configuration.parameter_tag),
    ("Manifest lists seven artifacts",             len(dataset_layout["artifacts"]) == 7),
    ("Manifest filenames match registry",          dataset_layout["artifacts"] == artifact_registry.artifact_filenames()),
    ("Every catalogued artifact on disk",          catalogued_artifacts_present),
]
for stage_9_description, stage_9_condition in stage_9_checks:
    print(f"[{'PASS' if stage_9_condition else 'FAIL'}] {stage_9_description}")

### Common Mistakes — Stage 9: Write the Dataset Layout Manifest

| Symptom | Likely Cause | How to Diagnose |
|---------|--------------|-----------------|
| Loader cannot find an artifact the manifest lists | manifest written before an artifact was saved | Assert every `artifacts` filename exists on disk (above) |
| Manifest tags drift from artifact names | tags edited between persistence and manifest write | Compare `dataset_layout['artifacts']` against `artifact_registry.artifact_filenames()` |
| `global_crop` disagrees with the focused products | crop mutated mid-run | Assert `global_crop == crop.as_tuple()` |
| Fewer than seven artifacts catalogued | a key dropped from `artifact_filenames` | Check `len(dataset_layout['artifacts']) == 7` |
| Manifest in the wrong directory | written to metadata instead of data dir | Confirm `dataset_layout_path.parent == data_directory` |

**Passing to the end-to-end summary:** every stage's contract holds in isolation. The final group runs the whole `ProcessingPipeline.run()` and confirms the orchestrated outputs agree with the stage-by-stage products assembled above.

---
## End-to-End Summary

`ProcessingPipeline.run()` executes the full ordered sequence — configuration dump, full tomogram, reduced tomogram, interferometric inputs, dataset layout — and returns a dictionary of the seven artifact paths plus the run directory. This group runs that single orchestrator call and asserts its outputs are consistent with the stage-by-stage products assembled above, then prints a structured summary table of every intermediate's shape, dtype and key statistics.

In [ ]:
end_to_end_configuration = ProcessingConfiguration(
    crop          = CropRegion(
        azimuth_start = processing_configuration.crop.azimuth_start,
        azimuth_end   = processing_configuration.crop.azimuth_end,
        range_start   = processing_configuration.crop.range_start,
        range_end     = processing_configuration.crop.range_end,
    ),
    input_configs = TomogramConfiguration(),
    parallel      = ParallelConfiguration(tomogram_workers=None, pyrat_threads=15),
    paths         = PathConfiguration(),
)

end_to_end_pipeline = ProcessingPipeline(end_to_end_configuration, logger=processing_logger)
end_to_end_artifacts = end_to_end_pipeline.run()

print("Returned artifact keys:")
for artifact_key, artifact_value in end_to_end_artifacts.items():
    print(f"  {artifact_key:<24}: {artifact_value}")

In [ ]:
expected_artifact_keys = {
    "tomogram_full", "tomogram_reduced", "dem_full", "dem_reduced",
    "primary_reduced", "secondaries_reduced", "interferograms_reduced", "run_directory",
}

end_to_end_paths_exist = all(
    Path(end_to_end_artifacts[key]).exists()
    for key in expected_artifact_keys if key != "run_directory"
)

summary_checks = [
    ("run() returned all expected keys",           set(end_to_end_artifacts.keys()) == expected_artifact_keys),
    ("Every returned artifact path exists",        end_to_end_paths_exist),
    ("Full tomogram name carries parameter_tag",   end_to_end_configuration.parameter_tag in Path(end_to_end_artifacts["tomogram_full"]).name),
    ("Reduced tomogram name carries tomogram_tag", end_to_end_configuration.tomogram_tag in Path(end_to_end_artifacts["tomogram_reduced"]).name),
    ("Interferograms name carries tomogram_tag",   end_to_end_configuration.tomogram_tag in Path(end_to_end_artifacts["interferograms_reduced"]).name),
    ("run_directory returned",                     Path(end_to_end_artifacts["run_directory"]).exists()),
]
for summary_description, summary_condition in summary_checks:
    print(f"[{'PASS' if summary_condition else 'FAIL'}] {summary_description}")

In [ ]:
intermediate_products = [
    ("full tomogram",     full_tomogram_array,        "elevation x azimuth x range, complex64"),
    ("full DEM",          full_dem_array,             "azimuth x range, real height [m]"),
    ("reduced tomogram",  reduced_tomogram_array,     "elevation x azimuth x range, complex64"),
    ("reduced DEM",       reduced_dem_array,          "azimuth x range, real height [m]"),
    ("primary SLC",       reloaded_primary,           "range x azimuth, complex64"),
    ("secondaries",       reloaded_secondaries,       "n_secondaries x range x azimuth, complex64"),
    ("interferograms",    reloaded_interferograms,    "n_secondaries x range x azimuth, complex64"),
]

print(f"{'product':<18}{'shape':<28}{'dtype':<14}{'|.| min':>12}{'|.| mean':>12}{'|.| max':>12}")
print("-" * 108)
for product_name, product_array, product_semantics in intermediate_products:
    product_magnitude = np.abs(product_array).astype(np.float64)
    print(f"{product_name:<18}{str(product_array.shape):<28}{str(product_array.dtype):<14}{product_magnitude.min():>12.4e}{product_magnitude.mean():>12.4e}{product_magnitude.max():>12.4e}")
print()
print("Semantic axis conventions:")
for product_name, _, product_semantics in intermediate_products:
    print(f"  {product_name:<18}: {product_semantics}")

### Interpretation

When every stage and the end-to-end group print only `PASS`, the processing pipeline produced a self-consistent set of raw TomoSAR products: two elevation-focused complex tomograms with their DEMs (the reduced one feeding the model, the full one supplying the parameter target), and a complex interferometric stack — all catalogued by `dataset.json` and reproducible from the frozen `config_state_<tag>.json`.

A `FAIL` localises the regression to a single stage: a tag mismatch points at variant resolution (Stage 2/5/6), a shape mismatch at crop subdivision or concatenation (Stage 2/4), a real-valued tomogram or interferogram at a lost-complex bug (Stage 4/7), out-of-range phase or saturated amplitude at the interferogram model (Stage 7), and a missing-artifact manifest at persistence ordering (Stage 8/9). Re-run this notebook before and after any change to a processing component: the assertions bracket the change and demonstrate behavioural equivalence rather than asserting it.